# Music Source Separation with Demucs (`htdemucs`)

This notebook uses Demucs (`htdemucs`) to separate **`data/music.mp3`** directly into the **`data/`** directory using relative paths:
- `data/vocals.wav`
- `data/drums.wav`
- `data/bass.wav`
- `data/other.wav`

> **Kernel**: Make sure the kernel is set to **`Python (seperate)`**.

## 1. Environment & GPU Setup

In [ ]:
from pathlib import Path
import os
import shutil
import torch
import demucs

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Demucs version : {demucs.__version__}")
print(f"Using device   : {device}")
if device == "cuda":
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

## 2. Configure Relative Paths & Input Info

In [ ]:
# Relative paths
input_audio = Path("data/music.mp3")
output_dir = Path("data")
model_name = "htdemucs"

assert input_audio.exists(), f"Input file not found at: {input_audio}"
input_size_mb = input_audio.stat().st_size / (1024 * 1024)
print(f"Input audio path : {input_audio} ({input_size_mb:.2f} MB)")
print(f"Output directory : {output_dir}")
print(f"Model name       : {model_name}")

## 3. Run Separation (Output Directly to `data/`)
Using `--filename "../{stem}.{ext}"` writes the output files directly into `data/` without creating extra subfolders.

In [ ]:
import demucs.separate

# Arguments for separation
cmd_args = [
    "-n", model_name,
    "-o", str(output_dir),
    "-d", device,
    "--filename", "../{stem}.{ext}",
    str(input_audio)
]

print(f"Running separation with args: {cmd_args}")
demucs.separate.main(cmd_args)

# Clean up empty htdemucs folder if created
extra_folder = output_dir / model_name
if extra_folder.exists() and not any(extra_folder.iterdir()):
    shutil.rmtree(extra_folder)

print("\nSeparation completed successfully!")

## 4. Verify Separated Stems in `data/`

In [ ]:
stems = ["vocals", "drums", "bass", "other"]

print(f"Separated files in {output_dir.resolve()}:")
for stem in stems:
    stem_file = output_dir / f"{stem}.wav"
    if not stem_file.exists():
        stem_file = output_dir / f"{stem}.mp3"
    
    if stem_file.exists():
        file_size_mb = stem_file.stat().st_size / (1024 * 1024)
        print(f"  ✓ {stem_file.name:<12} ({file_size_mb:.2f} MB) -> {stem_file}")
    else:
        print(f"  ✗ {stem_file.name:<12} (not found)")

## 5. (Alternative) Run via CLI
You can also run separation from the command line:

In [ ]:
!demucs -n htdemucs -o data --filename "../{stem}.{ext}" data/music.mp3